<a href="https://colab.research.google.com/github/camistrika/BETO_HUMOR/blob/main/notebooks/cross_validation_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validación cruzada (k-fold) — BETO + LoRA
Fija el learning_rate en el valor más estable visto en el grid search,
y explora combinaciones de `r` y `weight_decay` para elegir la mejor regularización.

In [1]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

Cloning into 'BETO_HUMOR'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 154 (delta 68), reused 154 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 3.11 MiB | 7.25 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/BETO_HUMOR
  Preparing metadata (setup.py) ... done


In [1]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.4 MB/s eta 0:00:00


In [2]:
%cd /content/BETO_HUMOR

/content/BETO_HUMOR


In [3]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer
from google.colab import files

from betohumor.config import DataConfig, BetoConfig, LoraConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.lora import build_beto_lora
from betohumor.hyperparam_search import run_one


## 1. Datos y configuraciones

In [ ]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)

# Unimos train+val para repartir en folds.
df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)
label_col = data_config.label_col

## 2. Configuración de la búsqueda


In [6]:
LR_VALUES     = [1e-4, 3e-4, 1e-3]
R_VALUES      = [8, 16, 32]
WEIGHT_DECAYS = [0.01, 0.05]
N_FOLDS       = 3

params_grid = [
    {'r': r, 'lora_alpha': r * 2, 'learning_rate': lr, 'weight_decay': wd}
    for lr, r, wd in product(LR_VALUES, R_VALUES, WEIGHT_DECAYS)
]


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
print(f'Total combinaciones: {len(params_grid)}')
params_grid

Total combinaciones: 18


[{'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0003, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0003, 'weight_decay': 0.05},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0003, 'weight_decay': 0.01},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0003, 'weight_decay': 0.05},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0003, 'weight_decay': 0.01},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0003, 'weight_decay': 0.05},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.001, 'weight_decay': 0.01},
 {'r'

## 3. Correr la validación cruzada

In [ ]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"r{params['r']}_lr{params['learning_rate']}_wd{params['weight_decay']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_lora/{run_name}"
        macro_f1, trainer = run_one(
            lambda p: build_beto_lora(beto_config, LoraConfig(r=p['r'], lora_alpha=p['lora_alpha'])),
            params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")



## 4. Resumen y guardado

In [9]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_lora.csv', index=False)

df_summary = (
    df_folds
    .groupby(['r', 'lora_alpha', 'learning_rate', 'weight_decay'])['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_lora_summary.csv', index=False)
df_summary

,r,lora_alpha,learning_rate,weight_decay,mean_macro_f1,std_macro_f1
15,32,64,0.0003,0.05,0.850698,0.006104
17,32,64,0.0010,0.05,0.850487,0.003123
6,16,32,0.0001,0.01,0.850293,0.008004
0,8,16,0.0001,0.01,0.850289,0.004723
13,32,64,0.0001,0.05,0.849713,0.000444
7,16,32,0.0001,0.05,0.849616,0.007783
10,16,32,0.0010,0.01,0.849565,0.002334
4,8,16,0.0010,0.01,0.849461,0.004124
14,32,64,0.0003,0.01,0.849376,0.003050
11,16,32,0.0010,0.05,0.849205,0.002832


In [10]:
files.download('results/cv_results_lora.csv')
files.download('results/cv_results_lora_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>